In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
folfox_out = pd.read_csv('../data/crc_folfox_out.csv', index_col=0)
folfiri_out = pd.read_csv('../data/crc_folfiri_out.csv', index_col=0)

folfox_out.rename(columns={'os_g_status': 'OS', 'pfs_m_g_status': 'PFS'}, inplace=True)
folfiri_out.rename(columns={'os_g_status': 'OS', 'pfs_m_g_status': 'PFS'}, inplace=True)


#add prefix to egfr_out columns except PFS and OS
folfox_out.columns = ['id_' + i if i not in ['PFS', 'OS'] else i for i in folfox_out.columns]
folfiri_out.columns = ['id_' + i if i not in ['PFS', 'OS'] else i for i in folfiri_out.columns]

In [3]:
folfox_out.head()

,id_record_id,id_institution,id_drugs_list,OS,id_tt_os_g_mos,PFS,id_tt_pfs_m_g_mos,id_sample_id
0,GENIE-DFCI-000233,DFCI,"Fluorouracil, Leucovorin Calcium, Oxaliplatin",0,100.986842,0.0,31.578947,GENIE-DFCI-000233-7633
1,GENIE-DFCI-000247,DFCI,"Fluorouracil, Leucovorin Calcium, Oxaliplatin",1,35.789474,0.0,3.914474,GENIE-DFCI-000247-11259
2,GENIE-DFCI-000306,DFCI,"bev, Fluorouracil, Leucovorin Calcium, Oxalipl...",0,38.125000,0.0,12.960526,GENIE-DFCI-000306-5914
3,GENIE-DFCI-000738,DFCI,"Fluorouracil, Leucovorin Calcium, Oxaliplatin",0,51.578947,NaN,NaN,GENIE-DFCI-000738-11390
4,GENIE-DFCI-000924,DFCI,"Fluorouracil, Leucovorin Calcium, Oxaliplatin",0,65.361842,NaN,NaN,GENIE-DFCI-000924-6164


In [4]:
mut = pd.read_csv('../data/crc_mutoh_pertreat.csv', index_col=0)
mut.columns = ['mut_' + i for i in mut.columns]
cna = pd.read_csv('../data/crc_cna_pertreat.csv', index_col=0)
cna.columns = ['cna_' + i for i in cna.columns]
fusion = pd.read_csv('../data/crc_fusion.csv', index_col=0)
fusion.columns = ['fus_' + i for i in fusion.columns]
clin = pd.read_csv('../data/crc_clin_pub.csv', index_col=0)
#add prefix to clin columns except PFS and OS
clin.columns = ['clin_' + i if i not in ['PFS', 'OS', 'OS_time'] else i for i in clin.columns]
#drop 'OS' and 'PFS' columns
clin = clin.drop(['OS', 'PFS'], axis=1)
cat_cols = [col for col in clin.columns if col.startswith('clin') and clin[col].nunique() <= 10]
num_cols = [col for col in clin.columns if col.startswith('clin') and clin[col].nunique() > 10]

In [5]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()

# le_dict = {}

# for i in cat_cols:
#     clin[i] = le.fit_transform(clin[i])
#     mapping = dict(zip(range(len(le.classes_)), le.classes_))
#     le_dict[i] = mapping
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='median')
for i in num_cols:
    clin[i] = imp.fit_transform(clin[i].values.reshape(-1,1))

In [6]:
print(mut.shape[1], cna.shape[1], fusion.shape[1], clin.shape[1])

224 224 256 22


In [7]:
print('total columns:', mut.shape[1] + cna.shape[1] + fusion.shape[1] + clin.shape[1])

total columns: 726


In [8]:
folfox = folfox_out.set_index('id_sample_id')
folfiri = folfiri_out.set_index('id_sample_id')


In [9]:
folfiri_mut = folfiri.join(mut, how='inner')
print(folfiri_mut.shape)
folfiri_mut_cna = folfiri_mut.join(cna, how='inner')
print(folfiri_mut_cna.shape)
folfiri_mut_cna_fus = folfiri_mut_cna.join(fusion, how='left')
fus_cols = [col for col in folfiri_mut_cna_fus.columns if col.startswith('fus')]
folfiri_mut_cna_fus[fus_cols] = folfiri_mut_cna_fus[fus_cols].fillna(0)
print(folfiri_mut_cna_fus.shape)


(936, 231)
(909, 455)
(909, 711)


In [10]:
folfox_mut = folfox.join(mut, how='inner')
folfox_mut_cna = folfox_mut.join(cna, how='inner')
folfox_mut_cna_fus = folfox_mut_cna.join(fusion, how='left')
fus_cols = [col for col in folfox_mut_cna_fus.columns if col.startswith('fus')]
folfox_mut_cna_fus[fus_cols] = folfox_mut_cna_fus[fus_cols].fillna(0)
print(folfox_mut_cna.shape)

(1290, 455)


In [11]:
folfiri_mut_cna_clin_fus = folfiri_mut_cna_fus.set_index('id_record_id').join(clin, how='inner')
folfox_mut_cna_clin_fus = folfox_mut_cna_fus.set_index('id_record_id').join(clin, how='inner')
print(folfox_mut_cna_clin_fus.shape)#, folfiri_mut_cna_clin_fus.shape)

(1290, 732)


In [12]:
folfox_mut_cna_clin_fus.head()

,id_institution,id_drugs_list,OS,id_tt_os_g_mos,PFS,id_tt_pfs_m_g_mos,mut_MAP3K1,mut_SPOP,mut_FLCN,mut_AURKA,...,clin_ca_first_dmets1,clin_ca_crc_td,clin_ca_crc_crm,clin_ca_crc_peri_inv,clin_crc_type,OS_time,clin_Histology Category,clin_Histology,clin_Derived Grade or Differentiation of Tumor,clin_CEA
GENIE-DFCI-000233,DFCI,"Fluorouracil, Leucovorin Calcium, Oxaliplatin",0,100.986842,0.0,31.578947,0,0,0,0,...,NaN,NaN,1.0,Not present/unknown,Rectal,3303,Adenocarcinoma,"Adenocarcinoma, NOS",II,1.0
GENIE-DFCI-000247,DFCI,"Fluorouracil, Leucovorin Calcium, Oxaliplatin",1,35.789474,0.0,3.914474,0,0,0,0,...,Liver,NaN,-1.0,Not present/unknown,Rectal,1163,Adenocarcinoma,"Adenocarcinoma, NOS",I,4.9
GENIE-DFCI-000306,DFCI,"bev, Fluorouracil, Leucovorin Calcium, Oxalipl...",0,38.125000,0.0,12.960526,0,0,0,0,...,Lung,NaN,1.0,Not present/unknown,Right colon,1230,Adenocarcinoma,"Adenocarcinoma, NOS",II,0.7
GENIE-DFCI-000738,DFCI,"Fluorouracil, Leucovorin Calcium, Oxaliplatin",0,51.578947,NaN,NaN,0,0,0,0,...,NaN,0.0,1.0,Not present/unknown,Right colon,1623,Adenocarcinoma,Mucinous,II,1.8
GENIE-DFCI-000924,DFCI,"Fluorouracil, Leucovorin Calcium, Oxaliplatin",0,65.361842,NaN,NaN,0,0,0,0,...,NaN,0.0,1.0,Perineural invasion present,Left colon,2039,Adenocarcinoma,"Adenocarcinoma, NOS",II,0.6


In [13]:
folfiri_mut_cna_clin_fus.to_csv('../data/crc_folfiri_mut_cna_fus_clin.csv')
folfox_mut_cna_clin_fus.to_csv('../data/crc_folfox_mut_cna_fus_clin.csv')